In [0]:
%spark.pyspark 

from pyspark.sql import functions as F

In [ ]:
%spark.pyspark

users = spark.createDataFrame(
[ 
    ("u1", "Berlin"),
    ("u2", "Berlin"),
    ("u3", "Munich"),
    ("u4", "Hamburg"), 
], ["user_id", "city"])

orders = spark.createDataFrame(
[ 
    ("o1", "u1", "p1", 2, 10.0),
    ("o2", "u1", "p2", 1, 30.0),
    ("o3", "u2", "p1", 1, 10.0),
    ("o4", "u2", "p3", 5, 7.0),
    ("o5", "u3", "p2", 3, 30.0),
    ("o6", "u3", "p3", 1, 7.0),
    ("o7", "u4", "p1", 10, 10.0),
], ["order_id", "user_id", "product_id", "qty", "price"])

products = spark.createDataFrame(
[
    ("p1", "Ring VOLA"),
    ("p2", "Ring POROG"),
    ("p3", "Ring TISHINA"), 
], ["product_id", "product_name"])

users.show()
orders.show()
products.show()

In [ ]:
%spark.pyspark

orders = orders.select("*", (F.col("qty") * F.col("price")).alias("revenue"))
orders.show()

In [ ]:
%spark.pyspark

joined = orders \
    .join(products, "product_id") \
    .join(users, "user_id")

joined.show()

In [ ]:
%spark.pyspark

metrics = joined.groupBy("city", "product_id", "product_name").agg(
        F.count("order_id").alias("orders_cnt"),
        F.sum("qty").alias("qty_sum"),
        F.sum("revenue").alias("revenue_sum"))

metrics.show()

In [ ]:
%spark.pyspark

from pyspark.sql.window import Window

window_city = Window.partitionBy("city").orderBy(F.desc("revenue_sum"))

top_revenue = metrics \
    .withColumn("place", F.row_number().over(window_city)) \
    .where(F.col("place") <= 2)

top_revenue.show()


In [ ]:
%spark.pyspark

paths = [
    "/tmp/sandbox_zeppelin/mart_city_top_products/",
    "s3a://sandbox-zeppelin/mart_city_top_products/"
]

for path in paths:
    top_revenue.write.mode("overwrite").parquet(path)

In [ ]:
%spark.pyspark

paths = [
    "/tmp/sandbox_zeppelin/mart_city_top_products/",
    "s3a://sandbox-zeppelin/mart_city_top_products/"
]

for path in paths:
    print(f"Reading from: {path}")
    spark.read.parquet(path).show()